# Part A: Word Embeddings

This notebook explores pre-trained word embeddings (Word2Vec and GloVe) using `gensim`.  
We will examine word similarities, analogies, and visualize embeddings with t-SNE.

## 1. Load Pre-trained Models

In [ ]:
!pip install gensim

In [ ]:
import gensim.downloader as api
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

sns.set_style("whitegrid")

# Load pre-trained models (this may take a few minutes on first run)
print("Loading Word2Vec (Google News 300d)...")
w2v_model = api.load('word2vec-google-news-300')
print("Done.")

print("Loading GloVe (Wiki Gigaword 300d)...")
glove_model = api.load('glove-wiki-gigaword-300')
print("Done.")

## 2. Top-10 Closest Words

Find and print the 10 most similar words for: **'car'**, **'jaguar'**, **'Jaguar'**, **'facebook'**.

In [ ]:
query_words = ['car', 'jaguar', 'Jaguar', 'facebook']

w2v_results = {}
glove_results = {}

for word in query_words:
    print(f"\n{'='*60}")
    print(f"  Word: '{word}'")
    print(f"{'='*60}")

    # Word2Vec
    try:
        w2v_similar = w2v_model.most_similar(word, topn=10)
        w2v_results[word] = [w for w, _ in w2v_similar]
        print(f"\n  Word2Vec top-10:")
        for rank, (w, score) in enumerate(w2v_similar, 1):
            print(f"    {rank:2d}. {w:<25s} {score:.4f}")
    except KeyError:
        print(f"  Word2Vec: '{word}' not in vocabulary")
        w2v_results[word] = []

    # GloVe
    try:
        glove_similar = glove_model.most_similar(word, topn=10)
        glove_results[word] = [w for w, _ in glove_similar]
        print(f"\n  GloVe top-10:")
        for rank, (w, score) in enumerate(glove_similar, 1):
            print(f"    {rank:2d}. {w:<25s} {score:.4f}")
    except KeyError:
        print(f"  GloVe: '{word}' not in vocabulary (GloVe is lowercase-only)")
        glove_results[word] = []

## 3. Common Words Between Word2Vec and GloVe Outputs

In [ ]:
for word in query_words:
    w2v_set = set(w.lower() for w in w2v_results.get(word, []))
    glove_set = set(w.lower() for w in glove_results.get(word, []))
    common = w2v_set & glove_set
    print(f"'{word}': {len(common)} common words -> {common if common else '{}'} ")

### **Your Observations:**

> **Q: Compare the Word2Vec and GloVe outputs. What differences do you notice? Why might they differ?**

*[Your answer here]*

> **Q: How do results differ for 'jaguar' vs 'Jaguar'? What does this tell us about case sensitivity in each model?**

*[Your answer here]*

> **Q: Comment on the results for 'facebook'. What type of training data might explain the outputs?**

*[Your answer here]*

## 4. Custom Words — Top-10 Closest

In [ ]:
# ============================================================
# CHANGE THESE 4 WORDS TO YOUR OWN CHOICES
# ============================================================
custom_words = ['python', 'coffee', 'galaxy', 'tennis']

for word in custom_words:
    print(f"\n{'='*60}")
    print(f"  Word: '{word}'")
    print(f"{'='*60}")

    for model_name, model_obj in [('Word2Vec', w2v_model), ('GloVe', glove_model)]:
        try:
            similar = model_obj.most_similar(word, topn=10)
            print(f"\n  {model_name} top-10:")
            for rank, (w, score) in enumerate(similar, 1):
                print(f"    {rank:2d}. {w:<25s} {score:.4f}")
        except KeyError:
            print(f"  {model_name}: '{word}' not in vocabulary")

### **Your Observations on Custom Words:**

> **Q: Comment on the results for your 4 custom words. Any interesting patterns?**

*[Your answer here]*

## 5. Analysis of 'student'

### 5a. Top-10 closest words to 'student'

In [ ]:
print("Top-10 closest words to 'student':\n")
for model_name, model_obj in [('Word2Vec', w2v_model), ('GloVe', glove_model)]:
    try:
        similar = model_obj.most_similar('student', topn=10)
        print(f"  {model_name}:")
        for rank, (w, score) in enumerate(similar, 1):
            print(f"    {rank:2d}. {w:<25s} {score:.4f}")
        print()
    except KeyError:
        print(f"  {model_name}: 'student' not in vocabulary")

### 5b. 'student' negatively correlated with "university students"

In [ ]:
negative_university = ['university', 'college', 'professor', 'degree', 'campus']

print("Top-10 closest to 'student' (negative: university/college context):\n")
for model_name, model_obj in [('Word2Vec', w2v_model), ('GloVe', glove_model)]:
    try:
        similar = model_obj.most_similar(positive=['student'], negative=negative_university, topn=10)
        print(f"  {model_name}:")
        for rank, (w, score) in enumerate(similar, 1):
            print(f"    {rank:2d}. {w:<25s} {score:.4f}")
        print()
    except KeyError as e:
        print(f"  {model_name}: KeyError – {e}")

### 5c. 'student' negatively correlated with primary/high school students

In [ ]:
negative_school = ['school', 'pupil', 'teacher', 'classroom', 'homework']

print("Top-10 closest to 'student' (negative: primary/high school context):\n")
for model_name, model_obj in [('Word2Vec', w2v_model), ('GloVe', glove_model)]:
    try:
        similar = model_obj.most_similar(positive=['student'], negative=negative_school, topn=10)
        print(f"  {model_name}:")
        for rank, (w, score) in enumerate(similar, 1):
            print(f"    {rank:2d}. {w:<25s} {score:.4f}")
        print()
    except KeyError as e:
        print(f"  {model_name}: KeyError – {e}")

### **Your Observations on 'student':**

> **Q: How do the results change when you negatively correlate with university vs. school context? What does this reveal about the embedding space?**

*[Your answer here]*

## 6. Word Analogies

Solve analogies using vector arithmetic: **a - b + c = ?**  
We report the top 2 closest words (excluding the input words).

In [ ]:
analogies = [
    ('king',     'man',     'woman'),     # king - man + woman = ?
    ('France',   'Paris',   'Tokyo'),     # France - Paris + Tokyo = ?
    ('trees',    'apples',  'grapes'),    # trees - apples + grapes = ?
    ('swimming', 'walking', 'walked'),    # swimming - walking + walked = ?
    ('doctor',   'father',  'mother'),    # doctor - father + mother = ?
]

def solve_analogy(model_obj, model_name, a, b, c, topn=2):
    """Solve: a - b + c = ? using vector arithmetic."""
    try:
        # most_similar with positive=[a, c] negative=[b] computes a - b + c
        results = model_obj.most_similar(positive=[a, c], negative=[b], topn=topn + 3)
        # Filter out the input words
        input_words = {a.lower(), b.lower(), c.lower()}
        filtered = [(w, s) for w, s in results if w.lower() not in input_words][:topn]
        return filtered
    except KeyError as e:
        return [(f"KeyError: {e}", 0.0)]

for a, b, c in analogies:
    print(f"\n{'='*60}")
    print(f"  {a} - {b} + {c} = ?")
    print(f"{'='*60}")

    for model_name, model_obj in [('Word2Vec', w2v_model), ('GloVe', glove_model)]:
        results = solve_analogy(model_obj, model_name, a, b, c)
        result_str = ', '.join([f"{w} ({s:.4f})" for w, s in results])
        print(f"  {model_name}: {result_str}")

### **Your Observations on Analogies:**

> **Q: Which model performs better on these analogies? Are there cases where one fails and the other succeeds?**

*[Your answer here]*

> **Q: Comment on the France - Paris + Tokyo analogy. Is the result correct and why?**

*[Your answer here]*

## 7. Custom Analogies

In [ ]:
# ============================================================
# DEFINE YOUR 3 CUSTOM ANALOGIES HERE
# Format: (a, b, c) => a - b + c = ?
# ============================================================
custom_analogies = [
    ('Madrid',  'Spain',   'Germany'),    # Madrid - Spain + Germany = ?
    ('cat',     'kitten',  'puppy'),      # cat - kitten + puppy = ?
    ('hot',     'summer',  'winter'),     # hot - summer + winter = ?
]

for a, b, c in custom_analogies:
    print(f"\n{'='*60}")
    print(f"  {a} - {b} + {c} = ?")
    print(f"{'='*60}")

    for model_name, model_obj in [('Word2Vec', w2v_model), ('GloVe', glove_model)]:
        results = solve_analogy(model_obj, model_name, a, b, c)
        result_str = ', '.join([f"{w} ({s:.4f})" for w, s in results])
        print(f"  {model_name}: {result_str}")

### **Your Observations on Custom Analogies:**

> **Q: Comment on the results of your 3 custom analogies. Do they produce the expected answers?**

*[Your answer here]*

## 8. t-SNE Visualization of GloVe Embeddings

Visualize the GloVe embeddings for 28 education/business-related words in 2D using t-SNE.

In [ ]:
tsne_words = [
    'business', 'career', 'student', 'university', 'college',
    'education', 'teacher', 'professor', 'school', 'degree',
    'economy', 'market', 'finance', 'investment', 'bank',
    'company', 'startup', 'entrepreneur', 'manager', 'salary',
    'science', 'research', 'technology', 'engineering', 'mathematics',
    'doctor', 'lawyer', 'accountant'
]

# Collect the embedding vectors
word_vectors = []
valid_words = []
for word in tsne_words:
    try:
        word_vectors.append(glove_model[word])
        valid_words.append(word)
    except KeyError:
        print(f"'{word}' not found in GloVe vocabulary, skipping.")

word_vectors = np.array(word_vectors)
print(f"Collected {len(valid_words)} word vectors of dimension {word_vectors.shape[1]}")

In [ ]:
# Run t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=8, n_iter=2000)
embeddings_2d = tsne.fit_transform(word_vectors)

# Plot
plt.figure(figsize=(14, 10))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c='steelblue', s=100, alpha=0.7, edgecolors='navy', linewidths=0.5)

for i, word in enumerate(valid_words):
    plt.annotate(word,
                 xy=(embeddings_2d[i, 0], embeddings_2d[i, 1]),
                 xytext=(7, 4),
                 textcoords='offset points',
                 fontsize=11,
                 fontweight='bold',
                 color='darkslategray')

plt.title('t-SNE Visualization of GloVe Embeddings (Education & Business Words)', fontsize=15, fontweight='bold')
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('tsne_glove_part_a.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as 'tsne_glove_part_a.png'")

### **Your Observations on t-SNE:**

> **Q: What word groupings/clusters do you observe in the t-SNE plot? Do they make semantic sense?**

*[Your answer here]*

> **Q: Are there any unexpected placements? How do education words relate to business words?**

*[Your answer here]*